# DR5 Prediction

## Dataset

In [ ]:
from utils import make_train_val_datasets,format_time
import time
import numpy as np
import torch

data_folder = ".\data\P\train"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

start_time = time.time()
train_ds_pre, val_ds_pre, global_stats = make_train_val_datasets(
    data_folder, 
    num_workers=1, 
    step_size=10, 
    window_size=30
)
print(f"Dataset Time: {format_time(time.time() - start_time)}")

np.savez("global_stats_P.npz", **global_stats)

lumped_size = train_ds_pre[0]['lumped'].shape[1]
point_size = train_ds_pre[0]['point'].shape[0]
print('lumped_size: ', lumped_size)
print('point_size: ', point_size)


## Training

In [ ]:
from utils import BatteryMFT, train_battery_model
import time

model = BatteryMFT(
    lumped_size=lumped_size, 
    point_size=point_size, 
    hidden_size=256, 
    encoder_method='lstm'   # best_method
).to(device)

start_train = time.time()

model = train_battery_model(
    model=model,
    save_dir=r"./model",
    train_dataset=train_ds_pre,
    val_dataset=val_ds_pre,
    mode='dr5_prediction',
    batch_size=64, 
    epochs=100,
    lr=1e-3, 
    device=device,
    patience=20,
    lambda_recon=0.1,
    lambda_soc=0.5
)

train_time = time.time() - start_train
print(f"Training Time: {format_time(train_time)}")


## Test

In [ ]:
from utils import BatteryMFT, evaluate_multitask, make_test_dataset_car_folder
import numpy as np
import torch


model_path = torch.load(r".\model\best_dr5_prediction.pth")
save_dir = r"./results"

stats = np.load("global_stats_P.npz", allow_pickle=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
data_folder = r".\data\P\test"

test_dataset = make_test_dataset_car_folder(
    folder_dir=data_folder, 
    global_stats=stats,
    car_type='P',
    step_size=10,
)

model = BatteryMFT(
    lumped_size=8,
    point_size=21,
    hidden_size=256,
    encoder_method='lstm'
)
model.load_state_dict(model_path)
model.to(device)

df = evaluate_multitask(
    model, test_dataset, device, stats,
    save_dir=save_dir
    )
